# Линейная регрессия

Практические ячейки к слайдам презентации. Заголовки секций совпадают с заголовками слайдов.


In [ ]:
# Setup
%matplotlib inline

import ipywidgets as widgets
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from ipywidgets import interact
from sklearn.compose import ColumnTransformer
from sklearn.datasets import fetch_california_housing, make_regression
from sklearn.impute import SimpleImputer
from sklearn.linear_model import HuberRegressor, Lasso, LinearRegression, QuantileRegressor, Ridge
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline, Pipeline
from sklearn.preprocessing import OneHotEncoder, PolynomialFeatures, StandardScaler

np.random.seed(42)


## Предсказание чисел `(viz)`


Простейшая линейная модель $y = kx + b$: подберём коэффициенты по данным и нарисуем прямую через облако точек.


In [ ]:
x = np.linspace(20, 120, 40)
y = 1.8 * x + 50 + np.random.randn(40) * 15
k, b = np.polyfit(x, y, 1)
x_line = np.linspace(x.min(), x.max(), 100)

plt.figure(figsize=(7, 4))
plt.scatter(x, y, color='steelblue', alpha=0.8, label='квартиры')
plt.plot(x_line, k * x_line + b, color='crimson', linewidth=2,
         label=f'y = {k:.2f}x + {b:.1f}')
plt.xlabel('площадь, м²')
plt.ylabel('цена, тыс. ₽')
plt.title('Линейная регрессия: прямая через облако точек')
plt.legend()
plt.tight_layout()
plt.show()


## Геометрическая интуиция и остатки `(viz)`


МНК минимизирует **вертикальные** расстояния от точек до линии. Остаток $\epsilon_i = y_i - \hat{y}_i$ — разница между фактом и прогнозом.


In [ ]:
x = np.linspace(0, 10, 25).reshape(-1, 1)
y = 1.5 * x.ravel() + 3 + np.random.randn(25) * 1.2

model = LinearRegression().fit(x, y)
y_hat = model.predict(x)

plt.figure(figsize=(7, 4))
plt.scatter(x, y, color='steelblue', label='данные')
plt.plot(x, y_hat, color='crimson', label='линия регрессии')
for i in range(0, len(x), 3):
    plt.plot([x[i, 0], x[i, 0]], [y[i], y_hat[i]], 'k--', alpha=0.6, linewidth=1)
plt.xlabel('признак x')
plt.ylabel('ответ y')
plt.title('Вертикальные остатки (пунктир)')
plt.legend()
plt.tight_layout()
plt.show()

residuals = y - y_hat
print(f'Сумма остатков (≈0 при intercept): {residuals.sum():.4f}')


## Функция потерь: почему MSE? `(experiment)`


Сравним MSE и MAE на одних данных: один выброс в $y$ сильно сдвигает линию МНК, но почти не влияет на медианную регрессию (MAE).


In [ ]:
x = np.linspace(0, 10, 30).reshape(-1, 1)
y = 2 * x.ravel() + 1 + np.random.randn(30) * 0.5

x_out = np.vstack([x, [[11.0]]])
y_out = np.append(y, 25.0)

mse_model = LinearRegression().fit(x_out, y_out)
mae_model = QuantileRegressor(quantile=0.5, alpha=0, solver='highs').fit(x_out, y_out)

x_plot = np.linspace(0, 12, 100).reshape(-1, 1)

plt.figure(figsize=(7, 4))
plt.scatter(x_out, y_out, color='steelblue', label='данные')
plt.plot(x_plot, mse_model.predict(x_plot), 'r-', label='MSE (LinearRegression)')
plt.plot(x_plot, mae_model.predict(x_plot), 'g--', label='MAE (медианная регрессия)')
plt.scatter([11], [25], color='orange', s=120, zorder=5, label='выброс')
plt.xlabel('x')
plt.ylabel('y')
plt.title('Влияние выброса на MSE vs MAE')
plt.legend()
plt.tight_layout()
plt.show()

print(f'Наклон MSE-модели: {mse_model.coef_[0]:.3f}')
print(f'Наклон MAE-модели: {mae_model.coef_[0]:.3f}')


## Как компьютер ищет минимум? `(example)`


`LinearRegression` решает МНК аналитически (линейная алгебра). `np.polyfit` для одного признака даёт тот же результат.


In [ ]:
x = np.linspace(0, 5, 20).reshape(-1, 1)
y = 3 * x.ravel() - 2 + np.random.randn(20) * 0.4

sk_model = LinearRegression().fit(x, y)
k_poly, b_poly = np.polyfit(x.ravel(), y, 1)

print('sklearn LinearRegression:')
print(f'  k = {sk_model.coef_[0]:.6f}, b = {sk_model.intercept_:.6f}')
print('np.polyfit:')
print(f'  k = {k_poly:.6f}, b = {b_poly:.6f}')
print(f'Разница k: {abs(sk_model.coef_[0] - k_poly):.2e}')


## Предобработка: масштабирование и категории `(example)`


Категории кодируем One-Hot, числовые признаки масштабируем — всё в одном `ColumnTransformer`.


In [ ]:
df = pd.DataFrame({
    'площадь': np.random.uniform(30, 120, 8),
    'этаж': np.random.randint(1, 20, 8),
    'район': np.random.choice(['центр', 'спальный', 'пригород'], 8),
})
df['цена'] = df['площадь'] * 1.5 + df['этаж'] * 2 + np.random.randn(8) * 5

X = df[['площадь', 'этаж', 'район']]
y = df['цена']

preprocessor = ColumnTransformer([
    ('num', StandardScaler(), ['площадь', 'этаж']),
    ('cat', OneHotEncoder(drop='first', sparse_output=False), ['район']),
])

X_transformed = preprocessor.fit_transform(X)
feature_names = preprocessor.get_feature_names_out()
print('Имена признаков после трансформации:')
print(list(feature_names))
print(f'\nФорма матрицы: {X_transformed.shape}')
print(f'Среднее числовых столбцов (≈0): {X_transformed[:, :2].mean(axis=0).round(3)}')


## Выбросы в данных `(viz)`


Выброс в $y$ сильно тянет обычную МНК-линию. `HuberRegressor` робастнее к таким точкам.


In [ ]:
x = np.linspace(0, 10, 25).reshape(-1, 1)
y = 1.2 * x.ravel() + 2 + np.random.randn(25) * 0.8

x_all = np.vstack([x, [[10.5]]])
y_all = np.append(y, 18.0)

ols = LinearRegression().fit(x_all, y_all)
huber = HuberRegressor().fit(x_all, y_all)

x_plot = np.linspace(0, 11, 100).reshape(-1, 1)

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
for ax, title, model, color in [
    (axes[0], 'Без выброса', LinearRegression().fit(x, y), 'steelblue'),
    (axes[1], 'С выбросом в y', None, 'steelblue'),
]:
    ax.scatter(x_all, y_all, color='steelblue')
    if title == 'Без выброса':
        ax.plot(x_plot, model.predict(x_plot), 'r-', label='OLS')
    else:
        ax.scatter([10.5], [18], color='orange', s=100, zorder=5, label='выброс')
        ax.plot(x_plot, ols.predict(x_plot), 'r-', label='OLS')
        ax.plot(x_plot, huber.predict(x_plot), 'g--', label='Huber')
        ax.legend()
    ax.set_title(title)
    ax.set_xlabel('x')
    ax.set_ylabel('y')
plt.tight_layout()
plt.show()


## Переобучение и разделение train/test `(experiment)`


Сравним $R^2$ на train и test: модель с большим числом признаков может «зазубрить» шум.


In [ ]:
X, y = make_regression(n_samples=80, n_features=15, n_informative=3,
                       noise=25, random_state=42)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42)

model = LinearRegression().fit(X_train, y_train)
r2_train = r2_score(y_train, model.predict(X_train))
r2_test = r2_score(y_test, model.predict(X_test))

print(f'R² на train: {r2_train:.3f}')
print(f'R² на test:  {r2_test:.3f}')
print(f'Разница:     {r2_train - r2_test:.3f}')


## Регуляризация: Ridge и Lasso `(viz)`


Ridge и Lasso сжимают веса. Сравним нормы коэффициентов OLS, Ridge и Lasso на данных с коррелированными признаками.


In [ ]:
X, y = make_regression(n_samples=100, n_features=8, n_informative=4,
                       random_state=42)
X[:, 1] = X[:, 0] + np.random.randn(100) * 0.05

models = {
    'OLS': make_pipeline(StandardScaler(), LinearRegression()),
    'Ridge': make_pipeline(StandardScaler(), Ridge(alpha=1.0)),
    'Lasso': make_pipeline(StandardScaler(), Lasso(alpha=0.1, max_iter=10000)),
}

fig, ax = plt.subplots(figsize=(8, 4))
labels, norms = [], []
for name, pipe in models.items():
    pipe.fit(X, y)
    coefs = pipe.named_steps[list(pipe.named_steps.keys())[-1]].coef_
    norm = np.linalg.norm(coefs)
    labels.append(name)
    norms.append(norm)
    ax.bar(np.arange(len(coefs)) + list(models.keys()).index(name) * 0.25,
           coefs, width=0.25, label=name)

ax.axhline(0, color='k', linewidth=0.5)
ax.set_xlabel('номер признака')
ax.set_ylabel('вес')
ax.set_title('Веса OLS vs Ridge vs Lasso')
ax.legend()
plt.tight_layout()
plt.show()

for name, norm in zip(labels, norms):
    print(f'||w||₂ для {name}: {norm:.3f}')


## Метрики качества регрессии `(example)`


Три стандартные метрики sklearn: MAE, RMSE и $R^2$.


In [ ]:
X, y = make_regression(n_samples=200, n_features=3, noise=15, random_state=42)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42)

model = LinearRegression().fit(X_train, y_train)
y_pred = model.predict(X_test)

mae = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2 = r2_score(y_test, y_pred)

print(f'MAE:  {mae:.2f}')
print(f'RMSE: {rmse:.2f}')
print(f'R²:   {r2:.3f}')


## Диагностика: графики остатков `(viz)`


График остатков: по оси X — предсказание $\hat{y}$, по оси Y — ошибка $y - \hat{y}$. Хорошо — случайное облако вокруг нуля.


In [ ]:
x = np.linspace(0, 10, 50).reshape(-1, 1)
y_good = 2 * x.ravel() + 1 + np.random.randn(50) * 1.5
y_bad = x.ravel() ** 2 + np.random.randn(50) * 3

lin_good = LinearRegression().fit(x, y_good)
lin_bad = LinearRegression().fit(x, y_bad)

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
for ax, y, model, title in [
    (axes[0], y_good, lin_good, 'Хорошо: линейная зависимость'),
    (axes[1], y_bad, lin_bad, 'Плохо: нелинейность не учтена'),
]:
    y_hat = model.predict(x)
    residuals = y - y_hat
    ax.scatter(y_hat, residuals, alpha=0.7)
    ax.axhline(0, color='crimson', linestyle='--')
    ax.set_xlabel('предсказание ŷ')
    ax.set_ylabel('остаток y - ŷ')
    ax.set_title(title)
plt.tight_layout()
plt.show()


## Интерпретация весов `(experiment)`


После `StandardScaler` веса сравнимы по силе влияния. Визуализируем коэффициенты на California Housing.


In [ ]:
housing = fetch_california_housing()
X, y = housing.data, housing.target
feature_names = housing.feature_names

pipe = make_pipeline(StandardScaler(), LinearRegression())
pipe.fit(X, y)
coefs = pipe.named_steps['linearregression'].coef_

order = np.argsort(np.abs(coefs))
plt.figure(figsize=(8, 5))
plt.barh(np.array(feature_names)[order], coefs[order], color='steelblue')
plt.xlabel('вес (после StandardScaler)')
plt.title('Интерпретация весов: California Housing')
plt.tight_layout()
plt.show()

for name, w in sorted(zip(feature_names, coefs), key=lambda t: abs(t[1]), reverse=True)[:3]:
    print(f'{name}: {w:+.3f}')


## sklearn и защита от утечек данных `(example)`


`Pipeline` гарантирует: imputer и scaler обучаются только на train, а к test применяется уже готовая трансформация.


In [ ]:
X, y = make_regression(n_samples=100, n_features=4, noise=10, random_state=42)
mask = np.random.rand(*X.shape) < 0.1
X = X.astype(float)
X[mask] = np.nan

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42)

pipe = Pipeline([
    ('imputer', SimpleImputer(strategy='mean')),
    ('scaler', StandardScaler()),
    ('reg', LinearRegression()),
])

pipe.fit(X_train, y_train)
score = pipe.score(X_test, y_test)
print(f'R² на test (Pipeline без утечки): {score:.3f}')
print('Шаги:', [name for name, _ in pipe.steps])


## sklearn и защита от утечек данных `(experiment)`


Если заполнить пропуски **до** `train_test_split`, статистики теста попадут в train — завышенное качество.


In [ ]:
X, y = make_regression(n_samples=120, n_features=5, noise=12, random_state=42)
mask = np.random.rand(*X.shape) < 0.15
X = X.astype(float)
X[mask] = np.nan

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42)

# Утечка: impute до split по всему X
imputer_leak = SimpleImputer(strategy='mean')
X_leak = imputer_leak.fit_transform(X)
X_tr_l, X_te_l, y_tr, y_te = train_test_split(
    X_leak, y, test_size=0.3, random_state=42)
model_leak = LinearRegression().fit(X_tr_l, y_tr)
r2_leak = r2_score(y_te, model_leak.predict(X_te_l))

# Без утечки: Pipeline
pipe = Pipeline([
    ('imputer', SimpleImputer(strategy='mean')),
    ('scaler', StandardScaler()),
    ('reg', LinearRegression()),
])
pipe.fit(X_train, y_train)
r2_safe = r2_score(y_test, pipe.predict(X_test))

print(f'R² с утечкой (impute до split): {r2_leak:.3f}')
print(f'R² без утечки (Pipeline):       {r2_safe:.3f}')


## Когда линейная модель не работает `(viz)`


При нелинейной зависимости прямая плохо аппроксимирует данные. Полиномиальные признаки помогают остаться в линейной модели.


In [ ]:
x = np.linspace(-3, 3, 60).reshape(-1, 1)
y = x.ravel() ** 2 + np.random.randn(60) * 1.5

lin = LinearRegression().fit(x, y)
poly = make_pipeline(PolynomialFeatures(degree=2), LinearRegression()).fit(x, y)

x_plot = np.linspace(-3, 3, 200).reshape(-1, 1)

plt.figure(figsize=(7, 4))
plt.scatter(x, y, color='steelblue', alpha=0.7, label='данные')
plt.plot(x_plot, lin.predict(x_plot), 'r-', label='линейная')
plt.plot(x_plot, poly.predict(x_plot), 'g--', label='PolynomialFeatures(2)')
plt.xlabel('x')
plt.ylabel('y')
plt.title('Нелинейные данные: прямая vs полином')
plt.legend()
plt.tight_layout()
plt.show()


## Предсказание чисел `(interactive)`


Подвигайте слайдеры $k$ и $b$ — наблюдайте, как меняется MSE (средний квадрат ошибки).


In [ ]:
x_data = np.linspace(1, 10, 25)
y_data = 2 * x_data + 3 + np.random.randn(25) * 1.5

def show_line(k=2.0, b=3.0):
    y_pred = k * x_data + b
    mse = np.mean((y_data - y_pred) ** 2)
    x_line = np.linspace(x_data.min(), x_data.max(), 100)
    plt.figure(figsize=(6, 4))
    plt.scatter(x_data, y_data, color='steelblue', label='данные')
    plt.plot(x_line, k * x_line + b, 'r-', label=f'y = {k:.1f}x + {b:.1f}')
    plt.title(f'MSE = {mse:.2f}')
    plt.xlabel('x')
    plt.ylabel('y')
    plt.legend()
    plt.show()

interact(show_line,
         k=widgets.FloatSlider(min=-1, max=5, step=0.1, value=2.0, description='k'),
         b=widgets.FloatSlider(min=-5, max=10, step=0.1, value=3.0, description='b'));


Покрутите слайдеры **k** и **b**, чтобы найти минимум MSE.


## Регуляризация: Ridge и Lasso `(interactive)`


Меняйте $\alpha$ в Ridge — смотрите, как сжимаются веса при росте штрафа.


In [ ]:
X, y = make_regression(n_samples=80, n_features=6, n_informative=3,
                       random_state=42)
X[:, 1] = X[:, 0] + np.random.randn(80) * 0.05
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

def show_ridge(alpha=1.0):
    model = Ridge(alpha=alpha).fit(X_scaled, y)
    coefs = model.coef_
    plt.figure(figsize=(7, 4))
    plt.bar(range(len(coefs)), coefs, color='steelblue')
    plt.axhline(0, color='k', linewidth=0.5)
    plt.xlabel('признак')
    plt.ylabel('вес')
    plt.title(f'Ridge: alpha={alpha:.2f}, ||w||₂={np.linalg.norm(coefs):.2f}')
    plt.show()

interact(show_ridge,
         alpha=widgets.FloatLogSlider(min=-3, max=3, step=0.1, value=1.0,
                                      description='alpha'));


Увеличьте **alpha** — веса станут меньше; при очень большом $\alpha$ все веса стремятся к нулю.
